# Step 3

# linear prediction model

In [3]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score , root_mean_squared_error
import joblib
import pickle

In [4]:
df = pd.read_csv("./cleaned_dataset.csv")
df["Date"] = pd.to_datetime(df["Date"], format="mixed")
df["day"] = df["Date"].dt.day

df = df.dropna(subset=["year","month","day"])

In [5]:
# labeliser les données
df["Service"] = df["Service"].astype("category").cat.codes
df["Gare de départ"] = df["Gare de départ"].astype("category").cat.codes
df["Gare d'arrivée"] = df["Gare d'arrivée"].astype("category").cat.codes

In [6]:
# Initialisation des X (les features) le y (la target)
X = df[["Gare de départ","Gare d'arrivée","Durée moyenne du trajet","month","year"]]
y = df["Retard moyen de tous les trains à l'arrivée"]

In [7]:
# Diviser les X (les features ) et y (valeurs que le modele predit)
X_train, X_test, y_train, y_test = train_test_split(X,y , random_state=104,test_size=0.15, shuffle=True)

In [8]:
#Modele lineaire
reg = LinearRegression()
reg.fit(X_train,y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [9]:
y_pred = reg.predict(X_test)
# evaluation du modele lineaires
R2 = r2_score(y_test,y_pred)
RMSE = root_mean_squared_error(y_test,y_pred)
MAE = mean_absolute_error(y_test,y_pred)
 
# print('X_train : ')
# print(X_train.head())
# print('')
# print('X_test : ')
# print(X_test.head())
# print('')
# print('y_train : ')
# print(y_train.head())
# print('')
# print('y_test : ')
# print(y_test.head())

In [10]:
# Baseline (reference minimal)
baseline_pred = np.full(len(y_test), y_train.mean())
baseline_RMSE = root_mean_squared_error(y_test, baseline_pred)
baseline_MAE = mean_absolute_error(y_test, baseline_pred)

In [11]:
# Sauvegarde dans un fichier pickle
with open("linear.pckl","wb") as p:
    pickle.dump(reg,p)

### Model Interpretation
I chose the Linear Regression model as a baseline because it is simple, interpretable, and allows us to understand the direct relationship between journey characteristics and delay duration.
### Features used:
Departure station, arrival station: the route directly influences delay likelihood
Average journey duration: longer journeys may accumulate more delays
Month and year: seasonal and temporal patterns affect punctuality

### Target: 
Average delay of all trains upon arrival was chosen because it best represents the passenger experience — what matters is arriving on time, not departing on time.
    
### Metrics results:
R²: the model explains X% of the variance in delays. A value close to 1 means the model predicts well, close to 0 means it struggles
RMSE: on average the model is off by X minutes, penalizing large errors more
MAE: on average the absolute error is X minutes, easier to interpret in real terms

### Limitations: 
Linear regression assumes a straight-line relationship between inputs and output. In reality, delays are influenced by complex non-linear factors such as strikes, weather, or infrastructure failures that a linear model cannot capture well. A tree-based model like Random Forest would likely perform better.


# tree-like prédiction modele (DecisionTreeRegressor)

In [23]:
%pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


##  Load the dataset into the variable 'df'

In [24]:
dfk = pd.read_csv('cleaned_dataset.csv')

## Display the first few rows to make sure it loaded correctly

In [25]:
#df.head()

# 1. Encode the text columns

In [26]:
le = LabelEncoder()
df['Service'] = le.fit_transform(df['Service'])
df['Gare de départ'] = le.fit_transform(df['Gare de départ'])
df["Gare d'arrivée"] = le.fit_transform(df["Gare d'arrivée"])

NameError: name 'LabelEncoder' is not defined

# 2. Handle missing values
# Dropping rows where the date features are missing to avoid scikit-learn errors

In [27]:
df = df.dropna(subset=['year', 'month', 'day'])

KeyError: ['day']

# Create Day of the Week

In [28]:
df['full_date'] = pd.to_datetime(df[['year', 'month', 'day']])
df['day_of_week'] = df['full_date'].dt.dayofweek
df = df.drop(columns=['full_date'])

KeyError: "['day'] not in index"

# 3. Define Features (X) and Target (y)
# Using the encoded categorical variables and date features as inputs

In [29]:
features = ['Service', 'Gare de départ', "Gare d'arrivée", 'year', 'month']
X = df[features]

# Predicting the continuous delay duration in minutes

In [30]:
y = df["Retard moyen de tous les trains à l'arrivée"]

# 4. Train/Test Split
# Creating training data and test data as outlined in your project flowchart

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Train the Model
# max_depth limits how deep the tree grows to help prevent overfitting

In [32]:
model = DecisionTreeRegressor(max_depth=10, random_state=42)
model.fit(X_train, y_train)

NameError: name 'DecisionTreeRegressor' is not defined

# 6. Evaluate the Model

In [33]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Decision Tree Model Evaluation ---")
print(f"RMSE: {rmse:.2f} minutes")
print(f"MAE:  {mae:.2f} minutes")
print(f"R²:   {r2:.2f}")

/home/hugo/Epitech/B-DAT-200-NCE-2-1-tardis-2/tardis/lib/python3.12/site-packages/sklearn/utils/validation.py:2684: UserWarning: X has feature names, but RandomForestRegressor was fitted without feature names
  warnings.warn(


ValueError: could not convert string to float: 'National'

# 7. Export the Model
# Saving the trained model file for dashboard integration

In [34]:
with open("DecisionTreeRegressor.pckl","wb") as m:
    pickle.dump(model,m)

## Model Interpretation - Decision Tree Regressor
### Model choice:
A Decision Tree Regressor was chosen because it naturally handles both categorical and numerical variables, is easy to interpret, and captures non-linear relationships between features and the target variable — which is important since train delays do not follow a simple linear pattern.
    
### Features used:

#### Service: 
the type of train service (National/International) may influence punctuality standards and route complexity
#### Gare de départ: 
the departure station affects delay likelihood due to differences in traffic volume and infrastructure
#### Gare d'arrivée:
the arrival station similarly influences delays, as some terminals are more congested than others
#### Year: 
captures long-term trends, such as infrastructure improvements or policy changes over time
#### Month: 
captures seasonal patterns, as delays tend to increase in winter due to weather conditions

### Target: 
The average delay of all trains upon arrival was chosen as it best reflects the passenger experience.
### Limitations:
A Decision Tree can easily overfit the training data, meaning it memorizes patterns instead of generalizing. This can result in good training scores but weaker performance on unseen data. To mitigate this, a maximum depth could be set, or a Random Forest (which combines many trees) could be used for better generalization.

# tree-like prédiction modele (Random Forest)

Objectif :
Construire un modèle de régression qui peut prédire le retard moyen
à l’arrivée à partir des caractéristiques du trajet.

In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import pickle

In [10]:
df = pd.read_csv("cleaned_dataset.csv")

if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

#df.head()

In [11]:
target = "Retard moyen de tous les trains à l'arrivée"
y = df[target]

In [12]:
features = df.drop(columns=[
    target,
    "Retard moyen des trains en retard à l'arrivée",
    "Retard moyen de tous les trains au départ"
], errors="ignore")

X = features

In [13]:
categorical_cols = X.select_dtypes(include="object").columns
numeric_cols = X.select_dtypes(include=np.number).columns

#categorical_cols, numeric_cols

In [14]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

In [15]:
model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [16]:
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", model)
])

In [17]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [18]:
print("NaN dans y :", y.isna().sum())
print("NaN dans X (total) :", X.isna().sum().sum())
nan_cols = X.columns[X.isna().any()].tolist()
print("Colonnes avec NaN :", nan_cols)

NaN dans y : 0
NaN dans X (total) : 236
Colonnes avec NaN : ['Gare de départ', "Gare d'arrivée", 'year', 'month']


In [19]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

In [20]:
y_pred = pipeline.predict(X_test)

In [21]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MAE :", mae)
print("RMSE :", rmse)
print("R² :", r2)

MAE : 1.0544634803077582
RMSE : 2.2430698078638827
R² : 0.759196352836119


In [22]:
with open("RandomForest.pckl", "wb") as m:
    pickle.dump(pipeline, m)

# Model Interpretation - Random Forest Regressor
### Model choice: 
A Random Forest Regressor was chosen as an improvement over a single Decision Tree. Instead of building one tree, it builds hundreds of trees on random subsets of the data and averages their predictions. This reduces overfitting and produces more stable, reliable predictions.

### Features used:
#### Date: the specific date of the journey
#### Service: the type of train service (National/International)
#### Gare de départ: the departure station affects delay likelihood due to differences in traffic volume and infrastructure
#### Gare d'arrivée: the arrival station influences delays as some terminals are more congested than others
#### Durée moyenne du trajet: longer journeys may accumulate more delays
#### Nombre de circulations prévues: the number of planned trains on a route affects congestion
#### Nombre de trains annulés: cancellations can indicate disruptions that also cause delays
#### Nombre de trains en retard au départ: trains already late at departure are likely to arrive late
#### Retard moyen des trains en retard au départ: the average departure delay directly impacts arrival delay
#### Nombre de trains en retard à l'arrivée: overall punctuality indicator for the route
#### Nombre trains en retard > 15min, > 30min, > 60min: distribution of delay severity
#### Retard moyen trains en retard > 15min: average delay for significantly late trains
#### Prct retard pour causes externes: weather, obstacles, external incidents
#### Prct retard pour cause infrastructure: track or signal failures
#### Prct retard pour cause gestion trafic: traffic management issues
#### Prct retard pour cause matériel roulant: rolling stock failures
#### Prct retard pour cause gestion en gare: station management and train reuse issues
#### Prct retard pour cause prise en compte voyageurs: passenger-related delays
#### Year: captures long-term trends over time
#### Month: captures seasonal patterns such as winter weather

### Target: 
The average delay of all trains upon arrival was chosen as it best reflects the passenger experience.
Advantages over Decision Tree: By averaging the predictions of 100 trees trained on different subsets of data, the Random Forest is much less likely to overfit. It is also more robust to noisy data and outliers, which is common in transport datasets where exceptional events like strikes or accidents can distort individual observations.

### Limitations: 
Random Forest is less interpretable than a single Decision Tree since it is harder to visualize and explain the combined decision of 100 trees. It is also slower to train and requires more memory. Sonnet 4.6